# Quantum Phase Estimation

QPE estimates the eigenvalue phase $\theta$ of a unitary $U|\psi\rangle = e^{2\pi i \theta}|\psi\rangle$.

We test with gates whose phases are known:
- **Z gate**: $\theta = 0.5$
- **S gate**: $\theta = 0.25$
- **T gate**: $\theta = 0.125$

Both a manual implementation and the built-in `ApplyQPE` are compared.

In [ ]:
import qsharp
import matplotlib.pyplot as plt

In [ ]:
qsharp.init()
with open("Program.qs") as f:
    qsharp.eval(f.read())

from qsharp.code import ManualEstimation, BuiltinEstimation

## Phase estimation results

Run both manual and built-in QPE for Z, S, and T gates at different precision levels.

In [ ]:
gates = {
    "Z": (0, 0.5),
    "S": (1, 0.25),
    "T": (2, 0.125),
}

precisions = [3, 4, 5]
shots = 50

for gate_name, (gate_type, expected) in gates.items():
    print(f"=== {gate_name} gate (expected phase: {expected}) ===")
    for precision in precisions:
        manual_results = [ManualEstimation(gate_type, precision) for _ in range(shots)]
        builtin_results = [BuiltinEstimation(gate_type, precision) for _ in range(shots)]
        manual_avg = sum(manual_results) / len(manual_results)
        builtin_avg = sum(builtin_results) / len(builtin_results)
        print(f"  precision={precision}: manual={manual_avg:.4f}, builtin={builtin_avg:.4f}")
    print()

## Distribution of estimates

Histogram of phase estimates for each gate using 4 bits of precision.

In [ ]:
precision = 4
shots = 200

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for col, (gate_name, (gate_type, expected)) in enumerate(gates.items()):
    for row, (method_name, method) in enumerate([("Manual", ManualEstimation), ("Built-in", BuiltinEstimation)]):
        ax = axes[row, col]
        results = [method(gate_type, precision) for _ in range(shots)]
        counts = {}
        for v in results:
            counts[v] = counts.get(v, 0) + 1

        sorted_items = sorted(counts.items())
        labels = [f"{k:.4f}" for k, _ in sorted_items]
        values = [v for _, v in sorted_items]
        colors = ["green" if abs(k - expected) < 1e-6 else "steelblue" for k, _ in sorted_items]
        ax.bar(labels, values, color=colors)
        ax.set_title(f"{method_name} - {gate_name} gate")
        ax.set_ylabel("Counts")
        ax.set_ylim(0, shots + 10)
        ax.axhline(y=0, color='gray', linewidth=0.5)

plt.suptitle(f"QPE Phase Distribution ({precision}-bit precision, {shots} shots)")
plt.tight_layout()
plt.show()